# Energy Drinks

Research question:  How do caffeine and sugar in energy drinks affect blood cortisol levels?

In [3]:
from tqdm.notebook import tqdm
import pandas as pd
from time import time_ns,sleep

from api.API import IslandsAPI
from api.tasks import * #Load all tasks
from random import shuffle, choice  # Method to randomize groups

tqdm.pandas() #Initialize tqdm


api = IslandsAPI() #Load api
participants = api.get_study_participants() #Get all participants who are contacts
shuffle(participants) #Shuffle the participants list

treatments = {
    "REGULAR" : ENERGY_DRINK,
    "CAFFEINE_FREE" : ENERGY_DRINK_CAFFEINE_FREE_250ML,
    "SUGAR_FREE" : ENERGY_DRINK_SUGAR_FREE_250ML,
    "CAFFEINE_AND_SUGAR_FREE" : ENERGY_DRINK_CAFFEINE_FREE_SUGAR_FREE_250ML
}

{'REGULAR': <api.tasks.Task at 0x1b3cf7c30b0>,
 'CAFFEINE_FREE': <api.tasks.Task at 0x1b3cf7f4c50>,
 'SUGAR_FREE': <api.tasks.Task at 0x1b3cf7f4cb0>,
 'CAFFEINE_AND_SUGAR_FREE': <api.tasks.Task at 0x1b3cf7f4c80>}

In [33]:
t_list = list(treatments.keys())

def rotate_t(l : list, n  : int) -> list:
    return l[ (-(n % len(l)) ) :] + l[:(-(n % len(l)) )]

assignment = pd.DataFrame([(p.get_id(),*rotate_t(t_list,i)) for (i,p) in enumerate(participants)], columns=["person_id", *[ f'treatment_{i+1}' for i in range(len(treatments))]])

assignment.to_csv('energy_drink_assignments.csv', index=False)



In [9]:
def wait_with_progress(endtime_ms : float, description: str):#Create a progress bar until a time is complete
    start = time_ns() // 1_000_000
    total_duration = endtime_ms - start

    with tqdm(total=total_duration, unit="ms", desc=description) as pbar:
        last_elapsed = 0

        while (now := time_ns() // 1_000_000) < endtime_ms:
            elapsed = now - start
            if elapsed > last_elapsed:
                pbar.update(elapsed - last_elapsed)
                last_elapsed = elapsed
            sleep(0.1)

        if last_elapsed < total_duration:
            pbar.update(total_duration - last_elapsed)

In [10]:
assignments = assignments.sample(frac=1) #Shuffle order of tasks

tqdm.pandas(desc="Starting Blood Pressure Task 1")
blood_pressure_init_1 = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood glucose levels

wait_with_progress(blood_pressure_init_1['end_time'].max(), 'Waiting for Blood Pressure Results 1')


tqdm.pandas(desc="Starting Blood Pressure Task 2")
blood_pressure_init_2 = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood glucose levels

wait_with_progress(blood_pressure_init_2['end_time'].max(), 'Waiting for Blood Pressure Results 2')

tqdm.pandas(desc='Retrieving Blood Pressure Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())
assignments['base_blood_pressure_1'] = assignments['person_id'].apply(lambda p: p.get_task_results()[1].result())

assignments['base_blood_pressure_2'] = assignments['person_id'].apply(lambda p: p.get_task_results()[0].result())
assignments.head()

Starting Blood Pressure Task 1:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Pressure Results 1:   0%|          | 0/57674.0 [00:00<?, ?ms/s]

Starting Blood Pressure Task 2:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Pressure Results 2:   0%|          | 0/58411.0 [00:00<?, ?ms/s]

Retrieving Blood Pressure Results:   0%|          | 0/48 [00:00<?, ?it/s]

,person_id,Temperature (C),Hydration,Salty Snack,base_blood_pressure_1,base_blood_pressure_2
14,Person: Id uyah6ege6y,-20,NoDrink,No Snack,130/85 mmHg,138/86 mmHg
33,Person: Id pblc6kgmtf,40,Drink,Snack,126/81 mmHg,126/82 mmHg
43,Person: Id 9ezaykx5zx,40,NoDrink,Snack,138/85 mmHg,132/80 mmHg
10,Person: Id rkm9rfff4p,-20,NoDrink,Snack,117/74 mmHg,123/73 mmHg
40,Person: Id t45ztfu6pd,40,NoDrink,Snack,124/81 mmHg,126/82 mmHg


In [11]:
def run_task(row, treatment_name):
    if row.isna()[treatment_name]:
        return None
    task : Task = treatments[treatment_name][str(row[treatment_name])]
    if task is None:
        return None
    return row['person_id'].do_task(task)

tqdm.pandas(desc='Starting Sit In Room Task')
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Temperature (C)'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Sitting in environment")



tqdm.pandas(desc="Starting Hydration")
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Hydration'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Hydrating")

tqdm.pandas(desc="Starting Salty Snack")
waiting_times = assignments.progress_apply(lambda row: run_task(row, 'Salty Snack'),axis=1).apply(pd.Series)
wait_with_progress(waiting_times['end_time'].max(), "Eating Snacks")

Starting Sit In Room Task:   0%|          | 0/48 [00:00<?, ?it/s]

Sitting in environment:   0%|          | 0/598367.0 [00:00<?, ?ms/s]

Starting Hydration:   0%|          | 0/48 [00:00<?, ?it/s]

Hydrating:   0%|          | 0/25706.699951171875 [00:00<?, ?ms/s]

Starting Salty Snack:   0%|          | 0/48 [00:00<?, ?it/s]

Eating Snacks:   0%|          | 0/152994.19995117188 [00:00<?, ?ms/s]

In [12]:
for i in range(0,num_subsamples):
    tqdm.pandas(desc=f'Measuring Blood Pressure {i+1}')
    blood_glucose_results = assignments['person_id'].progress_apply(lambda person: person.do_task(MEASURE_BLOOD_PRESSURE)).apply(pd.Series) #Have every person measure their blood pressure levels
    wait_with_progress(blood_glucose_results['end_time'].max(), f'Waiting for Blood Glucose Results {i+1}')


tqdm.pandas(desc='Retrieving Blood Pressure Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())

for i in range(0,num_subsamples):
    assignments[f'end_blood_pressure_{i+1}'] = assignments['person_id'].apply(lambda p: p.get_task_results()[i].result())

Measuring Blood Pressure 1:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Glucose Results 1:   0%|          | 0/51242.0 [00:00<?, ?ms/s]

Measuring Blood Pressure 2:   0%|          | 0/48 [00:00<?, ?it/s]

Waiting for Blood Glucose Results 2:   0%|          | 0/57875.0 [00:00<?, ?ms/s]

Retrieving Blood Pressure Results:   0%|          | 0/48 [00:00<?, ?it/s]

Data is then saved as a csv: [blood_pressure_results.csv](blood_pressure_results.csv)

In [13]:
assignments['person_id'] = assignments['person_id'].map(lambda x: x.get_id())

for i in range(0,num_subsamples):
    assignments[f'end_blood_pressure_{i+1}'] =  assignments[f'end_blood_pressure_{i+1}'].str.split(' ').map(lambda x: x[0]).astype(str) #Clean up rows so blood pressure is an integer


assignments['base_blood_pressure_1'] =  assignments['base_blood_pressure_1'].str.split(' ').map(lambda x: x[0]).astype(str) #Clean up rows so blood pressure is an integer
assignments['base_blood_pressure_2'] =  assignments['base_blood_pressure_2'].str.split(' ').map(lambda x: x[0]).astype(str) #Clean up rows so blood pressure is an integer
assignments.to_csv("blood_pressure_results.csv",index=False)